# Phase 1 — Data Preparation
Load, merge, and aggregate all CSV sources into one row per customer.

In [ ]:
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = "../data/raw/"

## 1. Load Data

### Customers

In [ ]:
customer_cols = ['cus_code', 'group_name__c', 'cus_name', 'Segment__c', 'Segment_Group__c', 'NumberOfEmployees', 'JobType', 'SalesPerson']
customers = pd.read_csv(os.path.join(DATA_DIR, "Customers.csv"), usecols=customer_cols)
print(f"Customers: {customers.shape[0]:,} rows — {customers.shape[1]} columns")
customers.head()

### Loads

In [ ]:
loads = pd.read_csv(
    os.path.join(DATA_DIR, "Loads.csv"),
    usecols=['cus_code', 'ode_net_amt', 'vou_label', 'vou_code', 'ord_posting_date']
)
loads['ord_posting_date'] = pd.to_datetime(loads['ord_posting_date'], utc=True)
loads = loads[loads['ord_posting_date'] >= '2023-01-01'].reset_index(drop=True)
print(f"Loads (2023+): {loads.shape[0]:,} rows — {loads.shape[1]} columns")
loads.head()

### Product

In [ ]:
product = pd.read_csv(os.path.join(DATA_DIR, "Product.csv"))
print(f"Product: {product.shape[0]:,} rows — {product.shape[1]} columns")
product

## 2. Enrich with Dimensions

In [ ]:
loads_enriched = (
    loads
    .merge(product[['vou_code', 'ProductFamily']], on='vou_code', how='left')
    .merge(customers, on='cus_code', how='left')
)

print(f"Enriched loads: {loads_enriched.shape[0]:,} rows — {loads_enriched.shape[1]} columns")
loads_enriched.head()

## 3. Monthly Aggregation per Customer & Product

In [ ]:
loads_enriched['month'] = loads_enriched['ord_posting_date'].dt.to_period('M')

groupby_cols = [
    'cus_code', 'group_name__c', 'cus_name',
    'Segment__c', 'NumberOfEmployees', 'JobType', 'SalesPerson',
    'ProductFamily',
    'month',
]

loads_monthly = (
    loads_enriched
    .groupby(groupby_cols, as_index=False)
    .agg(total_amt=('ode_net_amt', 'sum'), order_count=('ode_net_amt', 'count'))
    .sort_values(['cus_code', 'month'])
    .reset_index(drop=True)
)

print(f"Monthly aggregated: {loads_monthly.shape[0]:,} rows — {loads_monthly.shape[1]} columns")
loads_monthly.head()

## 4. Save Output

Export `loads_monthly` to `data/processed/` so Phase 2 can load it directly without re-running the full pipeline.

In [ ]:
PROCESSED_DIR = "../data/processed/"

output_path = os.path.join(PROCESSED_DIR, "loads_monthly.csv")
loads_monthly.to_csv(output_path, index=False)
print(f"Saved {len(loads_monthly):,} rows → {output_path}")